# Analisis Harga Cabai Indonesia 2026

**Harga cabai** adalah salah satu penggerak inflasi pangan yang paling
terasa: harganya bisa melonjak puluhan persen dalam hitungan hari.
Analisis ini memakai **data harian resmi PIHPS / Bank Indonesia**
(Pasar Tradisional) untuk 4 komoditas cabai di 34 provinsi, dari
2025-01-01 sampai 2026-10-16.

> **Sumber data: API publik PIHPS.** Data ditarik langsung dari API
> resmi PIHPS (hargapangan.id) dengan `scripts/fetch_pihps.py` — bukan
> salinan dataset pihak lain. Setiap angka di notebook ini dihitung
> dari data tersebut (lihat `findings.json` dan `sql/queries.sql`).
> Tidak ada angka yang dikarang.

> **Keterbatasan yang dicatat jujur.** (1) PIHPS melaporkan pada hari
> kerja pasar, bukan setiap hari kalender — jadi ada tanggal tanpa
> laporan. (2) PIHPS tidak mempublikasikan seri provinsi untuk
> 9 kombinasi provinsi x komoditas (misalnya
> Cabai Merah Besar di Sumatera Utara dan Sumatera Barat). Kombinasi
> tersebut dicatat sebagai gap di `findings.json`, tidak diisi angka
> lain dan tidak diinterpolasi.

## Pertanyaan analisis

1. Bagaimana harga 4 cabai bergerak di 2026, dan bagaimana dibanding 2025?
2. Cabai mana yang paling volatile?
3. Seberapa besar lonjakan Oktober 2026, terutama Cabai Rawit Merah?
4. Di provinsi mana cabai paling mahal dan paling murah?
5. Bagaimana harga di DKI Jakarta dibanding rata-rata nasional?


In [ ]:
import sqlite3
import pandas as pd
from IPython.display import Image, display

con = sqlite3.connect("data/harga_cabai.db")

def q(sql):
    return pd.read_sql(sql, con)


## 1. Cakupan data

Satu baris = satu hari pelaporan x satu komoditas x satu wilayah.


In [ ]:
q("SELECT komoditas, MIN(tanggal) AS awal, MAX(tanggal) AS akhir, COUNT(*) AS hari "
  "FROM harga_nasional GROUP BY komoditas")

## 2. Tren nasional 2025 vs 2026

Rata-rata nasional 2026: Cabai Rawit Merah Rp74.530/kg
(+13.3% vs 2025), Cabai Merah Keriting
Rp53.277/kg (-4.2%), Cabai Rawit Hijau
Rp56.578/kg (+10.7%), Cabai Merah Besar
Rp51.706/kg (-2.9%).


In [ ]:
q("SELECT komoditas, "
  "ROUND(AVG(CASE WHEN substr(tanggal,1,4)='2025' THEN harga_rp_per_kg END)) AS rata_2025, "
  "ROUND(AVG(CASE WHEN substr(tanggal,1,4)='2026' THEN harga_rp_per_kg END)) AS rata_2026 "
  "FROM harga_nasional WHERE harga_rp_per_kg IS NOT NULL GROUP BY komoditas")

In [ ]:
display(Image("charts/01_tren_nasional_4cabai.png"))

In [ ]:
display(Image("charts/08_banding_2025_2026.png"))

## 3. Volatilitas 2026

Cabai Merah Keriting adalah cabai paling volatile di 2026
(CV 20.0%), sedangkan
Cabai Rawit Hijau paling stabil
(CV 12.3%).


In [ ]:
display(Image("charts/03_volatilitas.png"))

In [ ]:
display(Image("charts/10_distribusi_harga.png"))

## 4. Lonjakan Oktober 2026

Cabai Rawit Merah menembus Rp100.000/kg pada
2026-09-30
(Rp100.550/kg).

Lalu menembus Rp105.000/kg pada
2026-10-12
(Rp105.300/kg).

Dari akhir September (Rp100.550/kg) ke puncak
Oktober (Rp105.300/kg pada 2026-10-12),
naik 4.7%. Kenaikan terbesar terjadi pada September
(MoM +39.3%), lalu Oktober naik lagi
+11.3%.

Fakta menarik: keempat komoditas mencapai puncak 2026 dalam jendela yang sama,
12–16 Oktober 2026 — lonjakannya serentak, bukan satu komoditas saja.


In [ ]:
display(Image("charts/05_lonjakan_oktober.png"))

In [ ]:
display(Image("charts/09_mom_rawit_merah.png"))

## 5. Disparitas antar provinsi 2026

Cabai Rawit Merah termahal di Maluku Utara
(rata-rata Rp113.213/kg), termurah di Sumatera Utara
(Rp46.494/kg) — selisih Rp66.719/kg
(143.5%).


In [ ]:
display(Image("charts/06_provinsi_termahal.png"))

In [ ]:
display(Image("charts/07_disparitas_provinsi.png"))

## 6. DKI Jakarta vs nasional

DKI Jakarta 2026: Cabai Rawit Merah
Rp78.792/kg, yaitu
+5.7% terhadap
rata-rata nasional.

## 7. Ke Power BI

Data yang sama disusun ulang menjadi star schema di `powerbi/`:
`fact_harga_cabai` + `dim_komoditas` + `dim_wilayah` + `dim_tanggal`,
15 measure DAX di `measures_dax.md`, layout 4 halaman di
`dashboard_layout.md`, dan panduan rakit di `build_guide.md`
(untuk Power BI Desktop di Windows).
